# Lab 0 Task 0.1 — CIFAR-10 CNN Experiments
This notebook implements a simple CNN for CIFAR-10 classification and compares different optimizers and activation functions as required for Task 0.1 of Lab 0.

## Setup & Imports

In [1]:
!nvidia-smi # Take a look at the GPU

Sun Mar 29 11:41:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 2080 Ti     On  |   00000000:1A:00.0 Off |                  N/A |
| 29%   23C    P8              1W /  250W |     502MiB /  11264MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import warnings
import wandb
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR10
from typing import List, Dict, Tuple, Any
from utils import make_loaders, train, validate, fit, evaluate

# Suppress NumPy 2.4 VisibleDeprecationWarning triggered inside torchvision
warnings.filterwarnings("ignore", category=np.exceptions.VisibleDeprecationWarning)

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Make results reproducible
torch.manual_seed(1)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(1)

2.11.0+cu130
True
NVIDIA GeForce RTX 2080 Ti
Using device: cuda


## Notebook parameters

In [3]:
LOG_WANDB = False # Notebook level parameter for enabling/disabling wandb logging
NUM_WORKERS = 8
PIN_MEMORY = True

## Data Loading

In [4]:
BATCH_SIZE = 256
VAL_FRACTION = 0.2  # 20% of training set used for validation

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

train_set = CIFAR10(root="../data", train=True,  download=True, transform=transform)
test_set  = CIFAR10(root="../data", train=False, download=True, transform=transform)

loader_kwargs = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

train_loader, val_loader, test_loader = make_loaders(
    train_set, test_set,
    loader_kwargs=loader_kwargs,
    val_fraction=VAL_FRACTION,
)

classes: list[str] = train_set.classes
print("Classes:", classes)

Dataset split — train: 40,000  val: 10,000  test: 10,000
Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


## Model Definition

| Layer | Detail |
|---|---|
| Conv Block 1 | Conv2d(3 → 32, 3×3, pad 1) → LeakyReLU → MaxPool2d |
| Conv Block 2 | Conv2d(32 → 64, 3×3, pad 1) → LeakyReLU → MaxPool2d |
| Conv Block 3 | Conv2d(64 → 128, 3×3, pad 1) → LeakyReLU → MaxPool2d |
| FC 1 | Linear(128 × 4 × 4 → 256) → LeakyReLU |
| FC 2 | Linear(256 → `num_classes`) |

The `activation` argument lets us swap between `LeakyReLU` and `Tanh` without rewriting the class.

In [5]:
class SimpleCNN(nn.Module):
    """
    A simple CNN for CIFAR-10 classification.

    Architecture:
        Conv Block 1 : Conv2d(3  → 32)  → act → MaxPool2d  (32×32 → 16×16)
        Conv Block 2 : Conv2d(32 → 64)  → act → MaxPool2d  (16×16 →  8×8)
        Conv Block 3 : Conv2d(64 → 128) → act → MaxPool2d  ( 8×8  →  4×4)
        FC 1         : Linear(128*4*4 → 256) → act
        FC 2         : Linear(256 → num_classes)
    """

    def __init__(
        self,
        num_classes: int = 10,
        activation: nn.Module = nn.LeakyReLU(),
    ) -> None:
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3,  32,  kernel_size=3, padding=1),
            activation,
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            activation,
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            activation,
            nn.MaxPool2d(2, 2),
        )

        self.classifier = nn.Sequential(
            nn.Linear(128 * 4 * 4, 256),
            activation,
            nn.Linear(256, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = torch.flatten(x, start_dim=1)
        return self.classifier(x)

## Experiment A – SGD + LeakyReLU
*(The original baseline from the script)*

In [6]:
# Hyperparameters
NUM_EPOCHS = 50
LEARNING_RATE = 1e-2

model_a = SimpleCNN(num_classes=len(classes), activation=nn.LeakyReLU()).to(device)
optimizer_a = optim.SGD(model_a.parameters(), lr=LEARNING_RATE)

wandb_kwargs = dict(
    entity="d7047e-group12",
    project="Lab0",
    name="SGD + LeakyReLU",
    tags=["Task 0.1"],
    config={"optimizer": "SGD", "lr": LEARNING_RATE, "activation": "LeakyReLU",
            "epochs": NUM_EPOCHS, "batch_size": BATCH_SIZE},
)

history_a = fit(
    model=model_a, 
    optimizer=optimizer_a,
    criterion=nn.CrossEntropyLoss(),
    train_loader=train_loader,
    val_loader=val_loader,
    wandb_kwargs=wandb_kwargs,
    num_epochs=NUM_EPOCHS,
    log=LOG_WANDB
)

Epoch 1/3 | train loss 2.2984, train acc 13.96% | val loss 2.2937, val acc 18.10%
Epoch 2/3 | train loss 2.2865, train acc 16.89% | val loss 2.2764, val acc 18.56%
Epoch 3/3 | train loss 2.2535, train acc 20.70% | val loss 2.2164, val acc 22.95%

Restored best weights (val loss 2.2164)


In [7]:
_ = evaluate(model=model_a, test_loader=test_loader,
         criterion=nn.CrossEntropyLoss(), label="SGD + LeakyReLU")

[SGD + LeakyReLU] Test loss: 2.2151 | Test acc: 22.99%


## Experiment B – Adam + LeakyReLU
*(Task: swap SGD for Adam and report accuracy)*

In [8]:
# Hyperparameters
NUM_EPOCHS = 50
LEARNING_RATE = 1e-3

model_b     = SimpleCNN(num_classes=len(classes), activation=nn.LeakyReLU()).to(device)
optimizer_b = optim.Adam(model_b.parameters(), lr=LEARNING_RATE)

wandb_kwargs = dict(
    entity="d7047e-group12",
    project="Lab0",
    name="Adam + LeakyReLU",
    tags=["Task 0.1"],
    config={"optimizer": "Adam", "lr": LEARNING_RATE, "activation": "LeakyReLU",
            "epochs": NUM_EPOCHS, "batch_size": BATCH_SIZE},
)

history_b = fit(
    model=model_b, 
    optimizer=optimizer_b,
    criterion=nn.CrossEntropyLoss(),
    train_loader=train_loader,
    val_loader=val_loader,
    wandb_kwargs=wandb_kwargs,
    num_epochs=NUM_EPOCHS,
    log=LOG_WANDB
)

Epoch 1/3 | train loss 1.6023, train acc 41.58% | val loss 1.3552, val acc 51.13%
Epoch 2/3 | train loss 1.2041, train acc 57.18% | val loss 1.1321, val acc 58.94%
Epoch 3/3 | train loss 1.0139, train acc 64.14% | val loss 1.0014, val acc 64.84%

Restored best weights (val loss 1.0014)


In [9]:
_ = evaluate(model=model_b, test_loader=test_loader,
         criterion=nn.CrossEntropyLoss(), label="Adam + LeakyReLU")

[Adam + LeakyReLU] Test loss: 0.9906 | Test acc: 65.10%


## Experiment C – Adam + Tanh
*(Task: swap LeakyReLU for Tanh and report accuracy)*

In [10]:
# Hyperparameters
NUM_EPOCHS = 50
LEARNING_RATE = 1e-3

model_c     = SimpleCNN(num_classes=len(classes), activation=nn.Tanh()).to(device)
optimizer_c = optim.Adam(model_c.parameters(), lr=LEARNING_RATE)

wandb_kwargs = dict(
    entity="d7047e-group12",
    project="Lab0",
    name="Adam + Tanh",
    tags=["Task 0.1"],
    config={"optimizer": "Adam", "lr": LEARNING_RATE, "activation": "Tanh",
            "epochs": NUM_EPOCHS, "batch_size": BATCH_SIZE},
)

history_c = fit(
    model=model_c, 
    optimizer=optimizer_c,
    criterion=nn.CrossEntropyLoss(),
    train_loader=train_loader,
    val_loader=val_loader,
    wandb_kwargs=wandb_kwargs,
    num_epochs=NUM_EPOCHS,
    log=LOG_WANDB
)

Epoch 1/3 | train loss 1.4679, train acc 47.54% | val loss 1.2490, val acc 55.21%
Epoch 2/3 | train loss 1.0557, train acc 62.86% | val loss 0.9789, val acc 65.09%
Epoch 3/3 | train loss 0.8746, train acc 69.30% | val loss 0.8974, val acc 68.17%

Restored best weights (val loss 0.8974)


In [11]:
_ = evaluate(model=model_c, test_loader=test_loader,
         criterion=nn.CrossEntropyLoss(), label="Adam + Tanh")

[Adam + Tanh] Test loss: 0.8792 | Test acc: 68.81%
